# Amber Index: Europe's wildfire protection gap

A 0–100 score per EU-27 country and year. **Higher = more fire danger relative to firefighting readiness.**
Bands: **Green** < 40 ≤ **Amber** < 60 ≤ **Red**.

| Pillar | Raises or lowers the score | Indicators |
|---|---|---|
| Danger | raises | burned area as % of land (GWIS via Our World in Data) |
| Hazards | raises | other hazards happening in the country: GDACS alerts (floods, earthquakes, droughts, storms, volcanoes, wildfires) per 100,000 km², weighted Green 0.5, Orange 3, Red 9 |
| Workforce | lowers | professional firefighters per 100k inhabitants (Eurostat LFS); volunteers if provided |
| Capability | lowers | fire-protection spending per inhabitant (Eurostat COFOG); fire stations per 1,000 km² and per 100k (OpenStreetMap) |

**Power** (0–100) estimates how much firefighting capacity a country can actually use: readiness (workforce and capability),
reduced by the hazard pressure it is under at the same time (danger and hazards): `power = 100 · readiness · (1 − α · pressure)`, α = 0.5.

All transformation logic lives in the `amber/` package, so the API applies exactly what this notebook trains.

## 1. Setup and configuration
Years, country list, cache behaviour and the random seed.

In [ ]:
import os, sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from amber import EU27, NAMES, SEED, SOURCES, data
from amber.features import build_panel, tidy_indicators, compute_rates, active_features, Preprocessor, FEATURES, PILLARS
from amber import scoring, model as M
from amber.power import add_power, ALPHA

REFRESH = False                     # True = download everything again
FIRST_YEAR, TRAIN_LAST = 2014, 2020 # scaling and imputation are fitted on 2014-2020 only
np.random.seed(SEED)
pd.set_option("display.width", 140, "display.max_columns", 20)

## 2. Fetch and cache every source
Each download is cached under `data/raw/` and reused unless `REFRESH=True`. OpenStreetMap counts come from the Overpass API with backoff and mirrors; countries that still fail stay missing.

In [ ]:
spending   = data.load_spending(REFRESH)
population = data.load_population(REFRESH)
area       = data.load_area(REFRESH)
burned     = data.load_burned_area(REFRESH)
firefighters = data.load_firefighters(REFRESH)
stations   = data.load_osm_stations(REFRESH)
volunteers = data.load_volunteers()
LAST_YEAR = int(min(burned.year.max(), population.year.max()))
YEARS = list(range(FIRST_YEAR, LAST_YEAR + 1))
hazard_events = data.load_hazard_events(FIRST_YEAR, LAST_YEAR, REFRESH)
hazards = data.hazard_load(hazard_events, YEARS)
print(f"Years {FIRST_YEAR}-{LAST_YEAR}; firefighter files cover {sorted(firefighters.year.unique())}")

In [ ]:
for name, df in [("spending", spending), ("population", population), ("area", area), ("burned area", burned),
                 ("firefighters", firefighters), ("OSM stations", stations), ("GDACS hazard events", hazard_events)]:
    print(f"--- {name}: {df.shape}"); display(df.head(3))

## 3. One tidy indicator table and one wide table
`indicators(iso2, year, indicator, value, source, imputed)` plus a wide country-year panel. Firefighter counts exist only for 2022–2024, so other years copy the nearest year and are flagged `imputed`; the OSM station snapshot is applied to every year.

In [ ]:
panel = build_panel(YEARS, spending, population, area, burned, firefighters, stations, volunteers, hazards)
indicators = tidy_indicators(panel)
display(indicators.sample(5, random_state=SEED))

In [ ]:
missing = (indicators.groupby(["iso2", "indicator"])["imputed"].mean().unstack().mul(100).round(0))
missing = missing.drop(columns=["volunteers"]) if panel.volunteers.isna().all() else missing
print("Share of years not directly observed (%), before model imputation")
display(missing.style.background_gradient(cmap="Greys", vmin=0, vmax=100).format("{:.0f}"))

## 4. Feature engineering
Per-capita and per-area rates → `log1p` (all are right-skewed) → winsorise at the 5th/95th percentile → min-max to 0–1. **The percentiles are fitted on 2014–2020 only** and stored in the preprocessor.

In [ ]:
rates = compute_rates(panel)
FEATS = active_features(panel)
meta = panel[["iso2", "year"]]
train_rows = panel.year <= TRAIN_LAST
pre = Preprocessor(FEATS).fit(rates.loc[train_rows, FEATS])
print("Features:", FEATS)
display(pd.DataFrame({"winsor_lo (log1p)": pre.lo, "winsor_hi (log1p)": pre.hi}).round(3))

## 5. Imputation
An `IterativeImputer`, fitted on the training years, fills what is still missing after scaling. The `imputed` mask records every value that was not observed.

In [ ]:
X = pre.transform(rates[FEATS])
imputed_mask = pre.missing_mask(rates)
imputed_mask["ff_per_100k"] |= panel["firefighters_imputed"].to_numpy()
print("Imputed share by feature (%):"); display(imputed_mask.mean().mul(100).round(1).to_frame("imputed %"))

## 6. Baseline index: equal weights
Equal weights inside each pillar and across the three pillars. Score = 100 · sigmoid(k · (Danger − Readiness − c)), with c the training median and k chosen so the 5th–95th percentile maps to about 5–95.

In [ ]:
W_EQ = scoring.equal_weights(FEATS)
scale_eq = scoring.fit_scale(scoring.gap(scoring.pillar_scores(X[train_rows], W_EQ), W_EQ))
base = scoring.score_table(X, meta, W_EQ, scale_eq)
base[base.year == LAST_YEAR].sort_values("amber", ascending=False).head(10).round(2)

In [ ]:
print("Bands, latest year:", base[base.year == LAST_YEAR]["band"].value_counts().to_dict())

## 7. Machine learning: what predicts next year's burned area?
Target: burned area (% of land) in year *t+1*. Features: readiness indicators and current danger at year *t*. ElasticNet baseline vs LightGBM (depth 3, 300 trees, monotone: more readiness can only lower the prediction, more danger can only raise it). A naive *persistence* forecast (next year = this year) is included as a reference. Rows are never shuffled.

In [ ]:
table = M.ml_table(rates[FEATS], meta)
cv = M.evaluate(table, FEATS, split_year=TRAIN_LAST)
cv.round(3)

In [ ]:
pick = cv.pivot(index="model", columns="validation", values="MAE")
best = pick.idxmin()
print("Lowest MAE per validation scheme:", best.to_dict())

### Final model and SHAP weights
The deployed LightGBM is trained on every year with a known target, using the preprocessor fitted on 2014–2020. Mean |SHAP| per feature gives weights within each pillar; each pillar's share of total |SHAP| gives the pillar weight. Final weights blend 50% ML and 50% equal.

In [ ]:
Xt = pre.transform(table[FEATS])
lgbm = M.make_lgbm(FEATS).fit(Xt, np.log1p(table["target_next"]))
W_ML, shap_values = M.shap_weights(lgbm, Xt)
W_FINAL = scoring.normalise(M.blend(W_ML, W_EQ, 0.5), FEATS)
weights_view = pd.DataFrame({"pillar": {f: FEATURES[f][0] for f in FEATS}, "w_ml": W_ML["within"],
                             "w_equal": W_EQ["within"], "w_final": W_FINAL["within"]})
pillar_view = pd.DataFrame({"w_ml": W_ML["pillar"], "w_equal": W_EQ["pillar"], "w_final": W_FINAL["pillar"]})
display(weights_view.round(3)); display(pillar_view.round(3))

## 8. Final scores and rank uncertainty
Scores with the blended weights. Monte Carlo: 500 runs with every weight perturbed by ±20%; `rank_lo`/`rank_hi` are the 5th/95th percentile ranks within each year.

In [ ]:
scale = scoring.fit_scale(scoring.gap(scoring.pillar_scores(X[train_rows], W_FINAL), W_FINAL))
scores = scoring.score_table(X, meta, W_FINAL, scale)
scores = scores.join(scoring.monte_carlo_ranks(X, meta, W_FINAL, scale, runs=500, jitter=0.2, seed=SEED))
print(f"k = {scale['k']:.2f}, c = {scale['c']:.3f}")
scores[scores.year == LAST_YEAR].sort_values("amber", ascending=False).head(10).round(2)

## 8b. Power: capacity left after concurrent hazards
Readiness (workforce and capability) reduced by the hazard pressure each country is under that year. A country with strong services but many simultaneous hazards has less power to spare.

In [ ]:
scores = add_power(scores, W_FINAL, ALPHA)
latest = scores[scores.year == LAST_YEAR].sort_values("power", ascending=False)
latest.assign(country=latest.iso2.map(NAMES))[["power_rank", "country", "power", "readiness", "hazard_pressure", "danger", "hazards"]].round(3).reset_index(drop=True)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(latest.iso2, latest.power, color=["#c2410c" if p > 0.5 else "#71717a" for p in latest.hazard_pressure])
ax.set_ylabel("power (0-100)"); ax.set_title(f"Power {LAST_YEAR}: orange = hazard pressure above 0.5"); plt.show()

## 9. Clustering
KMeans with 3 clusters on the pillar scores checks whether the band cut-offs separate real groups; KMeans with 4 clusters gives archetypes with plain-language names.

In [ ]:
from sklearn.cluster import KMeans
P = scores[[p for p in PILLARS if p in scores]]
k3 = KMeans(3, n_init=10, random_state=SEED).fit(P)
check = scores.assign(cluster=k3.labels_).groupby("cluster").agg(mean_amber=("amber", "mean"), n=("amber", "size"))
display(check.sort_values("mean_amber").round(1))
display(pd.crosstab(scores.assign(cluster=k3.labels_)["cluster"], scores["band"]))

In [ ]:
k4 = KMeans(4, n_init=10, random_state=SEED).fit(P)
labels = scoring.label_archetypes(pd.DataFrame(k4.cluster_centers_, columns=P.columns))
scores["archetype"] = [labels[c] for c in k4.labels_]
display(pd.DataFrame(k4.cluster_centers_, columns=P.columns).assign(archetype=labels).round(2))
scores[scores.year == LAST_YEAR].groupby("archetype")["iso2"].apply(lambda s: ", ".join(sorted(s)))

## 10. Visuals
Map of the latest year, the leaderboard, and which features drive the model.

In [ ]:
gdf = data.load_boundaries(REFRESH).merge(scores[scores.year == LAST_YEAR], on="iso2")
from matplotlib.colors import ListedColormap, BoundaryNorm
cmap, norm = ListedColormap(["#2f7d4f", "#d99a1c", "#c2410c"]), BoundaryNorm([0, 40, 60, 100], 3)
ax = gdf.to_crs(3035).plot(column="amber", cmap=cmap, norm=norm, edgecolor="white", linewidth=0.4, figsize=(8, 7), legend=False)
for _, r in gdf.to_crs(3035).iterrows():
    ax.annotate(f"{r.iso2}\n{r.amber:.0f}", r.geometry.representative_point().coords[0], ha="center", fontsize=7)
ax.set_title(f"Amber Index {LAST_YEAR}  (green < 40 ≤ amber < 60 ≤ red)"); ax.set_axis_off(); plt.show()

In [ ]:
board = scores[scores.year == LAST_YEAR].sort_values("amber", ascending=False)
board.assign(country=board.iso2.map(NAMES), rank_range=board.rank_lo.astype(str) + "–" + board.rank_hi.astype(str))[
    ["rank", "country", "amber", "band", "rank_range", "danger", "hazards", "workforce", "capability", "power", "archetype"]].round(2).reset_index(drop=True)

In [ ]:
import shap
shap.summary_plot(shap_values.to_numpy(), Xt, feature_names=[FEATURES[f][3] for f in FEATS], show=False, plot_size=(8, 4))
plt.title("What drives next year's burned area (LightGBM, SHAP)"); plt.tight_layout(); plt.show()

## 11. Save artifacts and prove the round-trip
Everything the API needs, written by `amber.model.save_artifacts`, then reloaded.

In [ ]:
ref = Xt.mean()   # the 'average country-year' that explanations are measured against
records = []
for idx, r in scores.iterrows():
    records.append({"iso2": r.iso2, "name": NAMES[r.iso2], "year": int(r.year),
        **{k: round(float(r[k]), 4) for k in ["danger", "hazards", "workforce", "capability", "readiness", "amber", "hazard_pressure", "power"]},
        "power_rank": int(r.power_rank),
        "hazard_events": {c.replace("events_", ""): int(panel.loc[idx, c]) for c in panel.columns if c.startswith("events_")},
        "band": r.band, "rank": int(r["rank"]), "rank_lo": int(r.rank_lo), "rank_hi": int(r.rank_hi), "archetype": r.archetype,
        "features": {f: round(float(X.loc[idx, f]), 4) for f in FEATS},
        "imputed": {f: bool(imputed_mask.loc[idx, f]) for f in FEATS},
        "firefighters_low_reliability": bool(panel.loc[idx, "firefighters_low_reliability"])})
weights_json = {f: {"pillar": FEATURES[f][0], "w_ml": W_ML["within"][f], "w_equal": W_EQ["within"][f], "w_final": W_FINAL["within"][f]} for f in FEATS}
weights_json["_pillars"] = {p: {"w_ml": W_ML["pillar"][p], "w_equal": W_EQ["pillar"][p], "w_final": W_FINAL["pillar"][p]} for p in PILLARS}
metadata = {"feature_order": FEATS, "training_years": [FIRST_YEAR, TRAIN_LAST], "model_years": [int(table.year.min()), int(table.year.max())],
            "latest_year": LAST_YEAR, "alpha": ALPHA, "cv_metrics": cv.round(4).to_dict(orient="records"), "scale": scale,
            "reference": {f: float(v) for f, v in ref.items()}, "bands": {"green_below": 40, "red_from": 60},
            "target": "burned area as % of land area in year t+1 (model predicts log1p)", "sources": SOURCES}
paths = M.save_artifacts(lgbm, pre, weights_json, records, metadata)
[p.name for p in paths]

In [ ]:
art = M.load_artifacts()
again = art["model"].predict(art["preprocessor"].transform(table[FEATS].head(5)))
assert np.allclose(again, lgbm.predict(Xt.head(5))), "round-trip mismatch"
print("Round-trip OK:", len(art["scores"]), "score rows,", art["metadata"]["feature_order"])